# TP Jour 2 — Modèle d'Exécution Spark, Jointures Multi-Datasets & Couche Silver (Delta Lake)

### Objectifs de la séance
1. Observer la **Lazy Evaluation** et distinguer transformations **Narrow** vs **Wide (Shuffle)** sur **Spark Web UI** (`http://localhost:4040`).
2. Profiler la qualité de vos tables **Bronze** (issues d'un ou **plusieurs datasets Kaggle**), nettoyer les anomalies, typer fortement les colonnes et harmoniser les **clés de jointure inter-sources**.
3. Mettre en œuvre une **jointure distribuée multi-datasets maîtrisée** (`BroadcastHashJoin` via `F.broadcast` pour un référentiel/dataset compact, ou `SortMergeJoin` entre deux grandes sources) **en contrôlant la cardinalité (absence d'explosion N-à-M)**.
4. Sauvegarder votre table analytique enrichie dans **`/lakehouse/silver/`** et expérimenter les garanties **ACID de Delta Lake** (`_delta_log`, **Schema Enforcement**, **`MERGE INTO`**, **Time Travel**).
5. Mener une **Analyse Exploratoire des Données (EDA)** et produire des **visualisations 1D, 2D et multidimensionnelles (ACP / PCA)** sur la couche Silver.

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from delta.tables import DeltaTable

spark = (
    SparkSession.builder
    .appName("J2_Spark_Delta_Silver")
    .getOrCreate()
)

print("Session Spark active — Spark UI : http://localhost:4040")

:: loading settings :: url = jar:file:/opt/venv/lib/python3.15/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /root/.ivy2.5.2/cache
The jars for the packages stored in: /root/.ivy2.5.2/jars
io.delta#delta-spark_4.2_2.13 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-929d8195-3f97-4b2b-a50b-995798d01ccc;1.0
	confs: [default]
	found io.delta#delta-spark_4.2_2.13;4.4.1 in central
	found io.delta#delta-storage;4.4.1 in central
	found io.unitycatalog#unitycatalog-client;0.6.0 in central
	found org.slf4j#slf4j-api;2.0.13 in central
	found org.apache.logging.log4j#log4j-slf4j2-impl;2.25.3 in central
	found org.apache.logging.log4j#log4j-api;2.25.3 in central
	found com.google.code.findbugs#jsr305;3.0.2 in central
	found io.unitycatalog#unitycatalog-hadoop;0.6.0 in central
	found org.apache.logging.log4j#log4j-core;2.25.3 in central
	found io.delta#delta-kernel-api;4.4.1 in central
	found

Session Spark active — Spark UI : http://localhost:4040


## 1. Chargement de vos tables Bronze (Multi-Datasets / Multi-Tables) & Diagnostic Qualité

### Exercice 1.1 — Charger toutes vos tables Bronze et profiler valeurs manquantes / doublons
1. Chargez dans `bronze_dfs` votre table de faits principale ainsi que la (ou les) table(s) secondaire(s) à joindre depuis `/lakehouse/bronze/<nom_table>`.
2. Comptez le nombre de valeurs `NULL` sur vos colonnes clés (en particulier les **clés de jointure**) ainsi que les doublons éventuels.

In [11]:
# --- 1. Découverte automatique des tables Delta Bronze sur HDFS (code fourni) ---
fs = spark._jvm.org.apache.hadoop.fs.FileSystem.get(spark._jsc.hadoopConfiguration())
PathHDFS = spark._jvm.org.apache.hadoop.fs.Path

bronze_dfs = {}
for s in sorted(fs.listStatus(PathHDFS("/lakehouse/bronze")), key=lambda x: x.getPath().getName()):
    name = s.getPath().getName()
    if s.isDirectory() and fs.exists(PathHDFS(s.getPath(), "_delta_log")):  # garde uniquement les tables Delta (exclut raw/)
        df = spark.read.format("delta").load(f"/lakehouse/bronze/{name}")
        bronze_dfs[name] = df
        print(f"[{name:<42}] -> {df.count():>9,} lignes | {df.columns[:5]}")

# --- 2. Diagnostic des NULL sur les dimensions (petites tables, complètes) ---
for name in ["airlines", "airports"]:
    df = bronze_dfs[name]
    cols = [c for c in df.columns if not c.startswith("_")]
    print(f"\n=== Valeurs nulles : {name} ===")
    df.select([F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in cols]).show(truncate=False)

# --- 3. Table principale : échantillon reproductible de 5 % des vols, mis en cache ---
SAMPLE_FRACTION = 0.05
df_bronze = bronze_dfs["flights"].sample(fraction=SAMPLE_FRACTION, seed=42).cache()
print(f"\nÉchantillon flights ({SAMPLE_FRACTION:.0%}) : {df_bronze.count():,} lignes")  # 1re action -> remplit le cache

# --- 4. Diagnostic des NULL sur l'échantillon (1 seul passage pour toutes les colonnes) ---
cols = [c for c in df_bronze.columns if not c.startswith("_")]
null_report = df_bronze.select([F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in cols])
assert null_report is not None, "TODO : Calculez le diagnostic de valeurs nulles `null_report`."
print("=== Valeurs nulles : flights (échantillon) ===")
null_report.show(truncate=False)

# --- 5. Doublons exacts (dropDuplicates = transformation wide -> shuffle) ---
n_dup = df_bronze.count() - df_bronze.dropDuplicates(cols).count()
print(f"Doublons exacts dans l'échantillon : {n_dup:,}")

[airlines                                  ] ->        14 lignes | ['iata_code', 'airline', '_ingested_at', '_source_dataset', '_source_file']
[airports                                  ] ->       322 lignes | ['iata_code', 'airport', 'city', 'state', 'country']
[flights                                   ] -> 5,819,079 lignes | ['year', 'month', 'day', 'day_of_week', 'airline']

=== Valeurs nulles : airlines ===
+---------+-------+
|iata_code|airline|
+---------+-------+
|0        |0      |
+---------+-------+


=== Valeurs nulles : airports ===
+---------+-------+----+-----+-------+--------+---------+
|iata_code|airport|city|state|country|latitude|longitude|
+---------+-------+----+-----+-------+--------+---------+
|0        |0      |0   |0    |0      |3       |3        |
+---------+-------+----+-----+-------+--------+---------+


Échantillon flights (5%) : 291,023 lignes


26/11/17 13:15:15 WARN CacheManager: Asked to cache already cached data.


=== Valeurs nulles : flights (échantillon) ===
+----+-----+---+-----------+-------+-------------+-----------+--------------+-------------------+-------------------+--------------+---------------+--------+----------+--------------+------------+--------+--------+---------+-------+-----------------+------------+-------------+--------+---------+-------------------+----------------+--------------+-------------+-------------------+-------------+
|year|month|day|day_of_week|airline|flight_number|tail_number|origin_airport|destination_airport|scheduled_departure|departure_time|departure_delay|taxi_out|wheels_off|scheduled_time|elapsed_time|air_time|distance|wheels_on|taxi_in|scheduled_arrival|arrival_time|arrival_delay|diverted|cancelled|cancellation_reason|air_system_delay|security_delay|airline_delay|late_aircraft_delay|weather_delay|
+----+-----+---+-----------+-------+-------------+-----------+--------------+-------------------+-------------------+--------------+---------------+--------+--

Doublons exacts dans l'échantillon : 0


## 2. Harmonisation Inter-Datasets, Nettoyage & Jointure Distribuée (`BroadcastHashJoin` / `SortMergeJoin`)

### Bonnes pratiques de jointure entre plusieurs datasets hétérogènes :
1. **Harmonisation des clés de jointure (*Key Alignment*)** : deux datasets Kaggle d'auteurs différents n'ont presque jamais exactement le même format de clé :
   - *Axe temporel* : ramenez les horodatages à la même granularité et au même fuseau (`F.to_date(...)`, `F.date_trunc("hour", ...)` ou format `yyyy-MM`).
   - *Axe géographique / catégoriel* : normalisez la casse, les espaces et les codes (`F.upper(F.trim(col))`, code ISO pays/région, code postal ou grille `F.round(lat, 2)`).
2. **Prévention des explosions N-à-M** : avant un `LEFT JOIN` de la table principale (`N` lignes) vers un second dataset, assurez-vous que la clé de jointure est **unique côté table secondaire** via une pré-agrégation métier (`groupBy(join_keys).agg(...)`) ou `dropDuplicates(join_keys)` ! Vérifiez systématiquement que le nombre de lignes après `LEFT JOIN` n'explose pas.
3. **Choix de la stratégie physique Spark** :
   - **Table secondaire compacte (< quelques dizaines de Mo)** : forcez **`F.broadcast(df_sec)`** pour obtenir un `BroadcastHashJoin` sans *Shuffle* de la grande table.
   - **Deux grands datasets volumineux** : laissez Spark exécuter un `SortMergeJoin` distribué (optimisé dynamiquement par *Adaptive Query Execution*).

### Exercice 2.1 — Nettoyer, harmoniser vos clés, joindre vos sources et écrire `/lakehouse/silver/<table_silver>`
1. Nettoyez vos tables (`dropDuplicates`, filtrage d'anomalies, typage fort `F.to_timestamp` / `F.to_date` / numériques) et harmonisez la clé de jointure commune.
2. Créez au moins **2 nouvelles colonnes métier dérivées** (dont idéalement un indicateur combinant vos deux sources !) utiles pour les Jours 3 & 4.
3. Réalisez la jointure (avec `F.broadcast(...)` sur une table de dimension/dataset secondaire compact), contrôlez le **taux de rapprochement (*match rate*)** ainsi que l'absence de duplication N-M, et affichez `.explain("simple")`.
4. Écrivez la table enrichie au format **Delta** dans **`/lakehouse/silver/<votre_table>`**.

#### Étape A — Profilage complet, nettoyage & typage de la table de faits `flights`
- Profilage sur le **dataset complet** pour détecter les anomalies rares que l'échantillon pourrait manquer.
- Nettoyage / typage sur l'**échantillon** (`df_bronze`), rapport qualité par règle, puis barrières `assert`.
- Choix : les vols aux codes aéroport non IATA (octobre 2015) sont **conservés** et signalés par `airport_code_valid`.

In [12]:
IATA_REGEX = "^[A-Z]{3}$"  # un code aéroport IATA = exactement 3 lettres majuscules

# =====================================================================
# 1. PROFILAGE SUR LE DATASET COMPLET (5,8 M lignes, 1 seul passage, sans cache)
#    -> détecter les anomalies rares que l'échantillon de 5 % aurait pu manquer
# =====================================================================
df_full = bronze_dfs["flights"]
key_cols = ["year", "month", "day", "airline", "flight_number", "origin_airport", "destination_airport", "scheduled_departure", "distance"]
range_cols = ["scheduled_departure", "scheduled_arrival", "distance", "scheduled_time", "departure_delay", "arrival_delay"]

full_profile = df_full.agg(
    F.count("*").alias("n_rows"),
    *[F.count(F.when(F.col(c).isNull(), c)).alias(f"null_{c}") for c in key_cols],
    *[F.min(c).alias(f"min_{c}") for c in range_cols],
    *[F.max(c).alias(f"max_{c}") for c in range_cols],
    F.sum((~F.col("origin_airport").cast("string").rlike(IATA_REGEX)).cast("int")).alias("n_origin_not_iata"),
    F.sum((~F.col("destination_airport").cast("string").rlike(IATA_REGEX)).cast("int")).alias("n_dest_not_iata"),
    F.sum((F.col("scheduled_departure") >= 2400).cast("int")).alias("n_sched_dep_ge_2400"),
    F.sum((F.col("distance") <= 0).cast("int")).alias("n_distance_le_0"),
).collect()[0].asDict()  # 1 seule ligne -> on la rapatrie en dictionnaire Python

print("=== Profil du dataset COMPLET (flights) ===")
for k, v in full_profile.items():
    print(f"  {k:<28} {v:>12,}" if v is not None else f"  {k:<28} {'NULL':>12}")

# =====================================================================
# 2. NETTOYAGE & TYPAGE (sur l'échantillon df_bronze) — transformations narrow uniquement
# =====================================================================
df_clean = (
    df_bronze
    # Date typée
    .withColumn("flight_date", F.make_date("year", "month", "day"))
    # Heure prévue HHMM : 5 -> "0005", puis timestamp. try_to_timestamp -> NULL si invalide (ex. 2400) au lieu de planter (mode ANSI)
    .withColumn("dep_hhmm", F.lpad(F.col("scheduled_departure").cast("string"), 4, "0"))
    .withColumn("scheduled_dep_ts", F.try_to_timestamp(
        F.concat_ws(" ", F.col("flight_date").cast("string"), F.col("dep_hhmm")),
        F.lit("yyyy-MM-dd HHmm")))
    .drop("dep_hhmm")
    # Booléens lisibles à la place des 0/1
    .withColumn("is_cancelled", F.col("cancelled") == 1)
    .withColumn("is_diverted", F.col("diverted") == 1)
    .drop("cancelled", "diverted")
    # Harmonisation des clés de jointure
    .withColumn("airline", F.upper(F.trim(F.col("airline"))))
    .withColumn("origin_airport", F.upper(F.trim(F.col("origin_airport").cast("string"))))
    .withColumn("destination_airport", F.upper(F.trim(F.col("destination_airport").cast("string"))))
    # Indicateur de validité du format des codes aéroport (option ii : on garde les lignes, on signale)
    .withColumn("airport_code_valid",
        F.col("origin_airport").rlike(IATA_REGEX) & F.col("destination_airport").rlike(IATA_REGEX))
)

# =====================================================================
# 3. RAPPORT QUALITÉ : combien de lignes chaque règle signale / va rejeter
# =====================================================================
quality = df_clean.agg(
    F.count("*").alias("n_total"),
    F.sum(F.col("scheduled_dep_ts").isNull().cast("int")).alias("n_invalid_ts"),
    F.sum((F.col("distance") <= 0).cast("int")).alias("n_distance_le_0"),
    F.sum((~F.col("airport_code_valid")).cast("int")).alias("n_airport_code_invalid"),
).collect()[0].asDict()

print("\n=== Rapport qualité (échantillon) ===")
for k, v in quality.items():
    print(f"  {k:<26} {v:>9,}  ({v / quality['n_total']:6.2%})")

# Rejet des lignes inexploitables (les codes aéroport invalides sont CONSERVÉS et signalés)
df_clean = df_clean.filter(F.col("scheduled_dep_ts").isNotNull() & (F.col("distance") > 0))

print("\nCodes aéroport invalides par mois :")
(df_clean.groupBy("month")
    .agg(F.count("*").alias("n_flights"),
         F.sum((~F.col("airport_code_valid")).cast("int")).alias("n_invalid_codes"))
    .orderBy("month")
    .show(12))

# =====================================================================
# 4. UNICITÉ DE LA CLÉ MÉTIER D'UN VOL (prépare le MERGE) — dropDuplicates = wide (shuffle)
# =====================================================================
flight_key = ["flight_date", "airline", "flight_number", "origin_airport", "scheduled_departure"]
n_clean = df_clean.count()
n_key_dup = n_clean - df_clean.dropDuplicates(flight_key).count()
print(f"Lignes après nettoyage : {n_clean:,} | doublons sur la clé métier {flight_key} : {n_key_dup:,}")

# =====================================================================
# 5. BARRIÈRES QUALITÉ (quality gates) : le notebook s'arrête si une anomalie imprévue apparaît
# =====================================================================
assert quality["n_invalid_ts"] / quality["n_total"] < 0.001, "Trop de timestamps invalides : nouvelle anomalie à investiguer !"
assert quality["n_distance_le_0"] / quality["n_total"] < 0.001, "Trop de distances <= 0 : nouvelle anomalie à investiguer !"
assert n_key_dup == 0, "Clé métier non unique : impossible de l'utiliser pour le MERGE !"
print("Barrières qualité OK")

=== Profil du dataset COMPLET (flights) ===
  n_rows                          5,819,079
  null_year                               0
  null_month                              0
  null_day                                0
  null_airline                            0
  null_flight_number                      0
  null_origin_airport                     0
  null_destination_airport                0
  null_scheduled_departure                0
  null_distance                           0
  min_scheduled_departure                 1
  min_scheduled_arrival                   1
  min_distance                           21
  min_scheduled_time                     18
  min_departure_delay                   -82
  min_arrival_delay                     -87
  max_scheduled_departure             2,359
  max_scheduled_arrival               2,400
  max_distance                        4,983
  max_scheduled_time                    718
  max_departure_delay                 1,988
  max_arrival_delay             


=== Rapport qualité (échantillon) ===
  n_total                      291,023  (100.00%)
  n_invalid_ts                       0  ( 0.00%)
  n_distance_le_0                    0  ( 0.00%)
  n_airport_code_invalid        24,381  ( 8.38%)

Codes aéroport invalides par mois :
+-----+---------+---------------+
|month|n_flights|n_invalid_codes|
+-----+---------+---------------+
|    1|    23442|              0|
|    2|    21720|              0|
|    3|    25207|              0|
|    4|    24272|              0|
|    5|    24893|              0|
|    6|    25162|              0|
|    7|    25954|              0|
|    8|    25554|              0|
|    9|    23148|              0|
|   10|    24381|          24381|
|   11|    23449|              0|
|   12|    23841|              0|
+-----+---------+---------------+



Lignes après nettoyage : 291,023 | doublons sur la clé métier ['flight_date', 'airline', 'flight_number', 'origin_airport', 'scheduled_departure'] : 0
Barrières qualité OK


#### Étape B — Colonnes métier dérivées sur la table de faits `flights`
- `dep_hour` : heure de départ prévue (0-23)
- `is_delayed_15` : retard à l'arrivée >= 15 min (norme officielle US) — NULL si vol non arrivé
- `delay_category` : cancelled / diverted / early / on_time / delayed / heavily_delayed
- `main_delay_cause` : cause de retard dominante (en minutes) parmi les 5 causes renseignées
- `route` : `ORIGIN-DEST`

In [13]:
# =====================================================================
# 1. CAUSE PRINCIPALE DU RETARD : la cause qui pèse le plus de minutes (ligne par ligne)
# =====================================================================
cause_cols = {
    "air_system_delay": "air_system",
    "security_delay": "security",
    "airline_delay": "airline",
    "late_aircraft_delay": "late_aircraft",
    "weather_delay": "weather",
}
max_cause = F.greatest(*[F.col(c) for c in cause_cols])  # max ligne par ligne (ignore les NULL)

# Construction d'une chaîne F.when(...).when(...) : du Python qui construit une expression Spark (rien n'est calculé ici)
main_cause_expr = None
for col_name, label in cause_cols.items():
    cond = F.col(col_name) == max_cause
    main_cause_expr = F.when(cond, label) if main_cause_expr is None else main_cause_expr.when(cond, label)

# =====================================================================
# 2. COLONNES MÉTIER (calculables sans l'autre source) — transformations narrow
# =====================================================================
df_features = (
    df_clean
    .withColumn("dep_hour", F.hour("scheduled_dep_ts"))
    .withColumn("is_delayed_15", F.col("arrival_delay") >= 15)  # NULL si annulé/dérouté (non arrivé)
    .withColumn("delay_category",
        F.when(F.col("is_cancelled"), "cancelled")          # en premier : arrival_delay est NULL pour ces vols
         .when(F.col("is_diverted"), "diverted")
         .when(F.col("arrival_delay") < 0, "early")
         .when(F.col("arrival_delay") < 15, "on_time")
         .when(F.col("arrival_delay") < 60, "delayed")
         .otherwise("heavily_delayed"))
    .withColumn("main_delay_cause", main_cause_expr)       # NULL si aucune cause renseignée
    .withColumn("route", F.concat_ws("-", "origin_airport", "destination_airport"))
)

# =====================================================================
# 3. CONTRÔLES : répartitions
# =====================================================================
print("Répartition des catégories de retard :")
df_features.groupBy("delay_category").count().orderBy(F.desc("count")).show()

print("Répartition des causes principales (vols avec retard >= 15 min) :")
df_features.filter(F.col("is_delayed_15")).groupBy("main_delay_cause").count().orderBy(F.desc("count")).show()

df_features.select("flight_date", "dep_hour", "route", "arrival_delay", "is_delayed_15", "delay_category", "main_delay_cause").show(8)

# =====================================================================
# 4. BARRIÈRES QUALITÉ
# =====================================================================
checks = df_features.agg(
    F.count("*").alias("n_total"),
    F.sum(F.col("delay_category").isNull().cast("int")).alias("n_category_null"),
    F.sum((F.col("is_cancelled") & (F.col("delay_category") != "cancelled")).cast("int")).alias("n_cancelled_misclassified"),
    F.sum(F.col("is_delayed_15").cast("int")).alias("n_delayed_15"),
    F.sum((F.col("is_delayed_15") & F.col("main_delay_cause").isNull()).cast("int")).alias("n_delayed_15_without_cause"),
).collect()[0].asDict()
print(checks)

assert checks["n_category_null"] == 0, "Des vols n'ont pas de delay_category !"
assert checks["n_cancelled_misclassified"] == 0, "Des vols annulés sont mal classés !"
assert checks["n_delayed_15_without_cause"] / checks["n_delayed_15"] < 0.01, "Trop de vols retardés >= 15 min sans cause renseignée !"
print("Barrières qualité étape B OK")

Répartition des catégories de retard :
+---------------+------+
| delay_category| count|
+---------------+------+
|          early|175199|
|        on_time| 57162|
|        delayed| 37012|
|heavily_delayed| 16357|
|      cancelled|  4561|
|       diverted|   732|
+---------------+------+

Répartition des causes principales (vols avec retard >= 15 min) :
+----------------+-----+
|main_delay_cause|count|
+----------------+-----+
|   late_aircraft|20699|
|         airline|15560|
|      air_system|15175|
|         weather| 1842|
|        security|   93|
+----------------+-----+

+-----------+--------+-------+-------------+-------------+--------------+----------------+
|flight_date|dep_hour|  route|arrival_delay|is_delayed_15|delay_category|main_delay_cause|
+-----------+--------+-------+-------------+-------------+--------------+----------------+
| 2015-01-01|       0|LAS-MCI|            6|        false|       on_time|            NULL|
| 2015-01-01|       5|BNA-ATL|            5|        fa

#### Étape C — Préparation des dimensions `airlines` et `airports`
- Clés harmonisées comme côté faits (`upper(trim(...))`), colonnes utiles uniquement, renommage sans conflit, types forcés.
- Barrière d'unicité des clés **avant** dédoublonnage (prévention de l'explosion N-M).
- *Role-playing dimension* : `airports` décliné en `dim_origin` (`origin_*`) et `dim_dest` (`dest_*`).
- Contrôle préalable (`left_anti`) des codes présents dans les vols mais absents des référentiels.

In [14]:
# =====================================================================
# 1. NETTOYAGE DES DIMENSIONS : clés harmonisées, colonnes utiles, noms sans conflit, types forcés
# =====================================================================
airlines_clean = bronze_dfs["airlines"].select(
    F.upper(F.trim("iata_code")).alias("airline_code"),   # même harmonisation que flights.airline
    F.trim("airline").alias("airline_name"),               # renommé : flights a déjà une colonne `airline` (le code)
)

airports_clean = bronze_dfs["airports"].select(
    F.upper(F.trim("iata_code")).alias("iata_code"),
    F.trim("airport").alias("airport_name"),
    F.trim("city").alias("city"),
    F.upper(F.trim("state")).alias("state"),
    F.col("latitude").cast("double").alias("lat"),
    F.col("longitude").cast("double").alias("lon"),
)  # country (toujours USA) et métadonnées _* volontairement exclues

# =====================================================================
# 2. BARRIÈRE : unicité de la clé AVANT dédoublonnage (détecter plutôt que masquer)
# =====================================================================
for name, df, key in [("airlines", airlines_clean, "airline_code"), ("airports", airports_clean, "iata_code")]:
    n, n_distinct = df.count(), df.select(key).distinct().count()
    print(f"{name:<9} : {n:>4} lignes | {n_distinct:>4} clés distinctes")
    assert n == n_distinct, f"Clé {key} non unique dans {name} : risque d'explosion N-M !"

# Filet de sécurité (sans effet ici puisque l'unicité est prouvée)
dim_airlines = airlines_clean.dropDuplicates(["airline_code"])
airports_clean = airports_clean.dropDuplicates(["iata_code"])

# =====================================================================
# 3. ROLE-PLAYING DIMENSION : 2 versions préfixées de la même table airports
# =====================================================================
airport_attrs = ["airport_name", "city", "state", "lat", "lon"]
dim_origin = airports_clean.select(
    F.col("iata_code").alias("origin_code"),
    *[F.col(c).alias(f"origin_{c}") for c in airport_attrs],
)
dim_dest = airports_clean.select(
    F.col("iata_code").alias("dest_code"),
    *[F.col(c).alias(f"dest_{c}") for c in airport_attrs],
)

dim_airlines.show(5, truncate=False)
dim_origin.show(5, truncate=False)

# =====================================================================
# 4. CONTRÔLE PRÉALABLE : codes présents dans les vols mais absents des dimensions (left_anti)
# =====================================================================
flight_airline_codes = df_features.select(F.col("airline").alias("airline_code")).distinct()
missing_airlines = flight_airline_codes.join(dim_airlines, on="airline_code", how="left_anti")

flight_airport_codes = (
    df_features.filter(F.col("airport_code_valid"))   # hors octobre (codes numériques déjà identifiés)
    .select(F.explode(F.array("origin_airport", "destination_airport")).alias("iata_code"))
    .distinct()
)
missing_airports = flight_airport_codes.join(airports_clean, on="iata_code", how="left_anti")

print(f"Compagnies des vols absentes de dim_airlines : {missing_airlines.count()} / {flight_airline_codes.count()}")
print(f"Aéroports (IATA) des vols absents de airports : {missing_airports.count()} / {flight_airport_codes.count()}")
missing_airports.show(truncate=False)

airlines  :   14 lignes |   14 clés distinctes
airports  :  322 lignes |  322 clés distinctes
+------------+---------------------------+
|airline_code|airline_name               |
+------------+---------------------------+
|AA          |American Airlines Inc.     |
|AS          |Alaska Airlines Inc.       |
|B6          |JetBlue Airways            |
|DL          |Delta Air Lines Inc.       |
|EV          |Atlantic Southeast Airlines|
+------------+---------------------------+
only showing top 5 rows
+-----------+-----------------------------------+-----------+------------+----------+----------+
|origin_code|origin_airport_name                |origin_city|origin_state|origin_lat|origin_lon|
+-----------+-----------------------------------+-----------+------------+----------+----------+
|ABE        |Lehigh Valley International Airport|Allentown  |PA          |40.65236  |-75.4404  |
|ABI        |Abilene Regional Airport           |Abilene    |TX          |32.41132  |-99.6819  |
|ABQ      

In [15]:
SILVER_MAIN_PATH = "/lakehouse/silver/flights_enriched"

# Étapes A -> C déjà réalisées : df_features (faits nettoyés + colonnes métier), dim_airlines / dim_origin / dim_dest (clés uniques)
count_before_join = df_features.count()

# =====================================================================
# 1. JOINTURES LEFT + BROADCAST des 3 dimensions (aucun shuffle de la table de faits)
# =====================================================================
df_silver = (
    df_features
    .join(F.broadcast(dim_airlines), F.col("airline") == F.col("airline_code"), "left")
    .join(F.broadcast(dim_origin), F.col("origin_airport") == F.col("origin_code"), "left")
    .join(F.broadcast(dim_dest), F.col("destination_airport") == F.col("dest_code"), "left")
    .drop("airline_code", "origin_code", "dest_code")
    # Indicateur combinant les 2 sources : vol intra-État (NULL si État inconnu, ex. octobre)
    .withColumn("is_same_state", F.col("origin_state") == F.col("dest_state"))
    # Traçabilité de la couche Silver
    .withColumn("_silver_processed_at", F.current_timestamp())
)

# =====================================================================
# 2. CONTRÔLES : cardinalité (pas d'explosion N-M) + match rate
# =====================================================================
count_after_join = df_silver.count()
print(f"Lignes avant jointure : {count_before_join:,} | après jointure : {count_after_join:,}")
assert count_after_join == count_before_join, "Explosion N-M ou perte de lignes lors de la jointure !"

match = df_silver.agg(
    F.avg(F.col("airline_name").isNotNull().cast("int")).alias("match_airline"),
    F.avg(F.col("origin_city").isNotNull().cast("int")).alias("match_origin_all"),
    F.avg(F.col("dest_city").isNotNull().cast("int")).alias("match_dest_all"),
    F.avg(F.when(F.col("airport_code_valid"), F.col("origin_city").isNotNull().cast("int"))).alias("match_origin_valid_codes"),
    F.avg(F.when(F.col("airport_code_valid"), F.col("dest_city").isNotNull().cast("int"))).alias("match_dest_valid_codes"),
).collect()[0].asDict()
print("Match rates :")
for k, v in match.items():
    print(f"  {k:<26} {v:7.2%}")

assert match["match_airline"] == 1.0, "Des compagnies n'ont pas été rapprochées !"
assert match["match_origin_valid_codes"] == 1.0 and match["match_dest_valid_codes"] == 1.0, \
    "Des codes IATA valides n'ont pas été rapprochés !"

assert df_silver is not None, "TODO : Construisez le DataFrame nettoyé et joint `df_silver`."
df_silver.explain("simple")  # attendu : 3 x BroadcastHashJoin, aucun Exchange hashpartitioning sur les vols

# =====================================================================
# 3. ÉCRITURE DELTA (déclenche tout le pipeline A -> D : lazy evaluation)
# =====================================================================
(df_silver.write.format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(SILVER_MAIN_PATH))

silver_count = spark.read.format("delta").load(SILVER_MAIN_PATH).count()
if count_before_join is not None:
    assert silver_count <= count_before_join, (
        f"Alerte explosion N-M : {silver_count:,} lignes après jointure > {count_before_join:,} lignes avant jointure !"
    )
print(f"Table Silver écrite dans {SILVER_MAIN_PATH} : {silver_count:,} lignes")

Lignes avant jointure : 291,023 | après jointure : 291,023


Match rates :
  match_airline              100.00%
  match_origin_all            91.62%
  match_dest_all              91.62%
  match_origin_valid_codes   100.00%
  match_dest_valid_codes     100.00%
== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Project [year#11625, month#11626, day#11627, day_of_week#11628, airline#16071, flight_number#11630, tail_number#11631, origin_airport#16072, destination_airport#16073, scheduled_departure#11634, departure_time#11635, departure_delay#11636, taxi_out#11637, wheels_off#11638, scheduled_time#11639, elapsed_time#11640, air_time#11641, distance#11642, wheels_on#11643, taxi_in#11644, scheduled_arrival#11645, arrival_time#11646, arrival_delay#11647, cancellation_reason#11650, air_system_delay#11651, ... 30 more fields]
   +- BroadcastHashJoin [destination_airport#16073], [dest_code#23425], LeftOuter, BuildRight, false, false
      :- Project [year#11625, month#11626, day#11627, day_of_week#11628, airline#16071, flight_number#11630, tail_num

Table Silver écrite dans /lakehouse/silver/flights_enriched : 291,023 lignes


## 3. Garanties Transactionnelles ACID de Delta Lake

### Exercice 3.1 — Explorer `_delta_log` et vérifier le *Schema Enforcement*
1. Lisez et affichez les premières lignes du fichier JSON de transaction `00000000000000000000.json` situé dans `{SILVER_MAIN_PATH}/_delta_log/`.
2. Tentez d'insérer (`mode("append")`) un petit DataFrame artificiel au schéma incompatible dans `SILVER_MAIN_PATH` et interceptez l'exception levée par Delta Lake.

In [16]:
import json

delta_log_dir = f"{SILVER_MAIN_PATH}/_delta_log"

def list_commits() -> list[str]:
    """Liste les fichiers de commit JSON (00000000000000000000.json, ...) du _delta_log sur HDFS."""
    return sorted(
        s.getPath().getName()
        for s in fs.listStatus(PathHDFS(delta_log_dir))
        if s.getPath().getName().endswith(".json")
    )

# =====================================================================
# 1. LECTURE BRUTE DU JOURNAL TRANSACTIONNEL (1 ligne JSON = 1 action)
# =====================================================================
spark.read.text(f"{delta_log_dir}/00000000000000000000.json").show(8, truncate=110)

# =====================================================================
# 2. LECTURE STRUCTURÉE : chaque type d'action devient une colonne
# =====================================================================
log_df = spark.read.json(f"{delta_log_dir}/00000000000000000000.json")
print("Types d'actions présents dans le commit 0 :", log_df.columns)

print("\n--- commitInfo : quelle opération, avec quels paramètres ---")
log_df.where(F.col("commitInfo").isNotNull()).select(
    "commitInfo.operation", "commitInfo.operationParameters", "commitInfo.operationMetrics"
).show(truncate=False)

print("--- metaData : le schéma de référence (utilisé par le Schema Enforcement) ---")
schema_str = log_df.where(F.col("metaData").isNotNull()).select("metaData.schemaString").first()[0]
schema_fields = json.loads(schema_str)["fields"]
print(f"{len(schema_fields)} colonnes, ex. : {[(f['name'], f['type']) for f in schema_fields[:6]]}")

print("\n--- add : les fichiers Parquet ajoutés, avec leurs statistiques (data skipping) ---")
log_df.where(F.col("add").isNotNull()).select(
    "add.path",
    (F.col("add.size") / (1024 * 1024)).cast("decimal(10,2)").alias("size_mb"),
    F.get_json_object("add.stats", "$.numRecords").alias("num_records"),
    F.get_json_object("add.stats", "$.minValues.arrival_delay").alias("min_arrival_delay"),
    F.get_json_object("add.stats", "$.maxValues.arrival_delay").alias("max_arrival_delay"),
).show(truncate=False)

# =====================================================================
# 3. SCHEMA ENFORCEMENT : 2 écritures invalides, toutes deux doivent être bloquées
# =====================================================================
commits_before = list_commits()

# Test 1 (TP) : schéma totalement différent
bad_df = spark.createDataFrame([("test_1", "colonne_intruse")], ["fake_id", "intruder_col"])
try:
    bad_df.write.format("delta").mode("append").save(SILVER_MAIN_PATH)
    raise AssertionError("Test 1 : l'écriture aurait dû être bloquée !")
except AssertionError:
    raise
except Exception as e:
    print("Test 1 — Schema Enforcement validé — écriture invalide bloquée :", str(e).splitlines()[0])

# Test 2 (réaliste) : une vraie ligne Silver + 1 colonne inattendue (pipeline modifié sans prévenir)
sneaky_df = (
    spark.read.format("delta").load(SILVER_MAIN_PATH)
    .limit(1)
    .withColumn("new_unexpected_col", F.lit("oops"))
)
try:
    sneaky_df.write.format("delta").mode("append").save(SILVER_MAIN_PATH)  # sans .option("mergeSchema", "true")
    raise AssertionError("Test 2 : l'écriture aurait dû être bloquée !")
except AssertionError:
    raise
except Exception as e:
    print("Test 2 — Schema Enforcement validé — colonne en trop bloquée :", str(e).splitlines()[0])

# =====================================================================
# 4. ATOMICITÉ : les écritures refusées n'ont laissé AUCUN commit
# =====================================================================
commits_after = list_commits()
print(f"\nCommits avant les tests : {commits_before}")
print(f"Commits après les tests : {commits_after}")
assert commits_after == commits_before, "Une écriture refusée a laissé une trace dans le _delta_log !"
print("Atomicité vérifiée : aucune version partielle créée")

+--------------------------------------------------------------------------------------------------------------+
|                                                                                                         value|
+--------------------------------------------------------------------------------------------------------------+
|{"commitInfo":{"timestamp":1794924752783,"operation":"WRITE","operationParameters":{"mode":"Overwrite","par...|
|{"metaData":{"id":"f2af59f5-196a-48e0-b428-55c3a1d63771","format":{"provider":"parquet","options":{}},"sche...|
|                                                      {"protocol":{"minReaderVersion":1,"minWriterVersion":2}}|
|{"add":{"path":"part-00003-4c1fe87b-2962-4415-8218-6863798f0dc8-c000.snappy.parquet","partitionValues":{},"...|
|{"add":{"path":"part-00007-9e8f0782-6b8b-43aa-8e5a-3aa81a973b4c-c000.snappy.parquet","partitionValues":{},"...|
+-----------------------------------------------------------------------------------------------

### Exercice 3.2 — Upsert (`MERGE INTO`) et Voyage dans le Temps (`VERSION AS OF`)
1. Prélevez un petit lot de lignes existantes dans votre table Silver (ou simulez un lot correctif + 1 nouvelle ligne) et appliquez un **`MERGE INTO`** via `DeltaTable.forPath(spark, SILVER_MAIN_PATH)` (`whenMatchedUpdate` / `whenNotMatchedInsert`).
2. Affichez l'historique des commits avec `DESCRIBE HISTORY` et interrogez la version initiale de la table via `.option("versionAsOf", 0)` pour démontrer l'auditabilité temporelle !

In [17]:
target_delta = DeltaTable.forPath(spark, SILVER_MAIN_PATH)

# Source construite depuis la VERSION 0 (Time Travel) -> mêmes vols à chaque exécution (cellule rejouable)
v0 = spark.read.format("delta").option("versionAsOf", 0).load(SILVER_MAIN_PATH)

# =====================================================================
# 1. LOT DE CORRECTIONS : 5 vols sans immatriculation (UPDATE) + 1 vol "oublié" (INSERT)
# =====================================================================
updates = (
    v0.filter(F.col("tail_number").isNull())
    .orderBy(*flight_key).limit(5)
    .withColumn("tail_number", F.lit("N000TP"))
)
new_flight = (
    v0.filter(F.col("tail_number").isNotNull())
    .orderBy(*flight_key).limit(1)
    .withColumn("flight_number", F.lit(9999))   # numéro de vol inédit -> nouvelle clé
)
assert new_flight.join(v0, on=flight_key, how="left_anti").count() == 1, "Le vol 9999 existe déjà en version 0 !"

source = updates.unionByName(new_flight)
print("Lot de corrections (source du MERGE) :")
source.select(*flight_key, "tail_number").show(truncate=False)

# =====================================================================
# 2. MERGE INTO (upsert) sur la clé métier — 1 seule transaction = 1 seul commit
# =====================================================================
merge_cond = " AND ".join(f"t.{k} = s.{k}" for k in flight_key)
print("Condition du MERGE :", merge_cond)

(target_delta.alias("t")
    .merge(source.alias("s"), merge_cond)
    .whenMatchedUpdate(set={
        "tail_number": "s.tail_number",
        "_silver_processed_at": "current_timestamp()",
    })
    .whenNotMatchedInsertAll()
    .execute())

# =====================================================================
# 3. HISTORIQUE DES COMMITS
# =====================================================================
history_df = spark.sql(f"DESCRIBE HISTORY delta.`{SILVER_MAIN_PATH}`")
history_df.select("version", "timestamp", "operation", "operationMetrics").show(truncate=80)

last_merge = history_df.filter(F.col("operation") == "MERGE").orderBy(F.desc("version")).first()
m = last_merge["operationMetrics"]
n_upd, n_ins = int(m["numTargetRowsUpdated"]), int(m["numTargetRowsInserted"])
print(f"Dernier MERGE (version {last_merge['version']}) : {n_upd} ligne(s) mise(s) à jour, {n_ins} insérée(s)")
# 1re exécution : 5 + 1 | exécutions suivantes : 6 + 0 (le vol 9999 existe déjà -> mis à jour, pas dupliqué)
assert n_upd + n_ins == 6, "Le MERGE n'a pas traité exactement les 6 lignes du lot !"

# =====================================================================
# 4. TIME TRAVEL : version 0 vs version courante
# =====================================================================
current = spark.read.format("delta").load(SILVER_MAIN_PATH)
n_v0_sql = spark.sql(f"SELECT COUNT(*) FROM delta.`{SILVER_MAIN_PATH}` VERSION AS OF 0").first()[0]
print(f"\nLignes en version 0 : {n_v0_sql:,} | version courante : {current.count():,}")

print("Mêmes vols, deux versions de la table :")
(source.select(*flight_key)
    .join(v0.select(*flight_key, F.col("tail_number").alias("tail_number_v0")), on=flight_key, how="left")
    .join(current.select(*flight_key, F.col("tail_number").alias("tail_number_current")), on=flight_key, how="left")
    .orderBy("flight_number")
    .show(truncate=False))

# =====================================================================
# 5. BARRIÈRE : la clé métier reste unique après le MERGE (pas de doublon créé)
# =====================================================================
n_current = current.count()
assert n_current == current.dropDuplicates(flight_key).count(), "Le MERGE a créé des doublons sur la clé métier !"
print("Clé métier toujours unique après le MERGE")

assert history_df.count() >= 2, "TODO : Exécutez au moins une opération MERGE/UPDATE pour créer une version >= 1 !"

Lot de corrections (source du MERGE) :
+-----------+-------+-------------+--------------+-------------------+-----------+
|flight_date|airline|flight_number|origin_airport|scheduled_departure|tail_number|
+-----------+-------+-------------+--------------+-------------------+-----------+
|2015-01-01 |F9     |1257         |MIA           |1050               |N000TP     |
|2015-01-02 |US     |1806         |LAX           |915                |N000TP     |
|2015-01-03 |F9     |1070         |ORD           |730                |N000TP     |
|2015-01-03 |US     |724          |DCA           |925                |N000TP     |
|2015-01-03 |US     |763          |CLT           |1655               |N000TP     |
|2015-01-01 |AA     |9999         |DFW           |755                |N371AA     |
+-----------+-------+-------------+--------------+-------------------+-----------+

Condition du MERGE : t.flight_date = s.flight_date AND t.airline = s.airline AND t.flight_number = s.flight_number AND t.origin_ai

26/11/17 15:08:38 WARN MapPartitionsRDD: RDD 808 was locally checkpointed, its lineage has been truncated and cannot be recomputed after unpersisting


+-------+-----------------------+---------+--------------------------------------------------------------------------------+
|version|              timestamp|operation|                                                                operationMetrics|
+-------+-----------------------+---------+--------------------------------------------------------------------------------+
|      1|2026-11-17 15:08:37.055|    MERGE|{numTargetRowsCopied -> 270648, numTargetRowsDeleted -> 0, numTargetBytesRemo...|
|      0|2026-11-17 14:12:32.811|    WRITE|{numFiles -> 3, numRemovedFiles -> 0, numRemovedBytes -> 0, numOutputRows -> ...|
+-------+-----------------------+---------+--------------------------------------------------------------------------------+

Dernier MERGE (version 1) : 5 ligne(s) mise(s) à jour, 1 insérée(s)



Lignes en version 0 : 291,023 | version courante : 291,024
Mêmes vols, deux versions de la table :


+-----------+-------+-------------+--------------+-------------------+--------------+-------------------+
|flight_date|airline|flight_number|origin_airport|scheduled_departure|tail_number_v0|tail_number_current|
+-----------+-------+-------------+--------------+-------------------+--------------+-------------------+
|2015-01-03 |US     |724          |DCA           |925                |NULL          |N000TP             |
|2015-01-03 |US     |763          |CLT           |1655               |NULL          |N000TP             |
|2015-01-03 |F9     |1070         |ORD           |730                |NULL          |N000TP             |
|2015-01-01 |F9     |1257         |MIA           |1050               |NULL          |N000TP             |
|2015-01-02 |US     |1806         |LAX           |915                |NULL          |N000TP             |
|2015-01-01 |AA     |9999         |DFW           |755                |NULL          |N371AA             |
+-----------+-------+-------------+-----------